# Piloto de factibilidad: harness de inventarios (IIC-3694)

Este cuaderno mide si las tareas caen en la "zona aprendible" del modelo objetivo.

**Antes de empezar**
1. Menú *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4*.
2. Sube `inventory_harness_colab.zip` a `MyDrive/inventory_harness/` en tu Google Drive.

**Cómo usarlo.** Ejecuta las celdas en orden. Solo la celda de la sección 4 tiene valores que puedes cambiar.

La sección 7 (piloto) no corre hasta que la actives a propósito, así que *Ejecutar todo* es seguro: llega hasta la exploración y su análisis.

**Si Colab se desconecta:** vuelve a ejecutar las secciones 1 a 4 y luego la etapa que estaba corriendo. Los resultados están en Drive y la corrida continúa sin duplicar. Puede perderse la última muestra escrita, que se vuelve a generar.

## 1. Verificar la GPU

In [ ]:
import torch

if torch.cuda.is_available():
    GPU = torch.cuda.get_device_name(0)
    memoria = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"GPU detectada: {GPU} ({memoria:.1f} GiB)")
else:
    GPU = None
    print("=" * 70)
    print("ATENCION: NO HAY GPU. En CPU esto tardaria muchas horas.")
    print("Activa la GPU: Entorno de ejecucion > Cambiar tipo de entorno > T4.")
    print("Luego vuelve a ejecutar el cuaderno desde esta celda.")
    print("=" * 70)

## 2. Montar Google Drive

Colab pedirá permiso para acceder a tu Drive. Todos los resultados quedan en `MyDrive/inventory_harness/`.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/inventory_harness")
BASE.mkdir(parents=True, exist_ok=True)
RESULTADOS = BASE / "results"
RESULTADOS.mkdir(exist_ok=True)
ZIP_CODIGO = BASE / "inventory_harness_colab.zip"

if ZIP_CODIGO.exists():
    print("Zip encontrado:", ZIP_CODIGO)
else:
    print(f"FALTA EL ZIP. Sube inventory_harness_colab.zip a {BASE} y repite esta celda.")

## 3. Instalar dependencias y descomprimir el proyecto

No se reinstala `torch`: se usa el que trae Colab, compilado para su GPU. El código se descomprime en el disco local, cada vez desde cero, para no mezclar versiones. Al final corren las pruebas del núcleo, que deben pasar (no usan GPU).

In [ ]:
!pip install -q "transformers>=4.51" accelerate pytest

In [ ]:
import os
import shutil
import sys
import zipfile

CODIGO = Path("/content/inventory_harness")
shutil.rmtree(CODIGO, ignore_errors=True)
with zipfile.ZipFile(ZIP_CODIGO) as archivo:
    archivo.extractall(CODIGO)
sys.path.insert(0, str(CODIGO / "src"))
os.chdir(CODIGO)
print("Proyecto en", CODIGO)

In [ ]:
!python -m pytest -q tests/test_solver.py tests/test_numbers_es.py tests/test_verifier.py tests/test_generator.py

## 4. Configuración

**Esta es la única celda que puedes editar.**

- Todos los modelos tienen máximo 2B de parámetros. Qwen y Llama no necesitan token.
- Gemma es opcional: requiere aprobación en Hugging Face y un token. Si lo activas, guarda el token en Colab Secrets (icono de llave, a la izquierda) con el nombre `HF_TOKEN` y dale acceso a este cuaderno. Un modelo sin acceso se registra y se salta; el resto sigue.
- El token no se imprime nunca.
- `BATCH_SIZE = None` usa 16 para cada modelo. Si falta memoria, el lote se reduce solo.

In [ ]:
MODELO_OBJETIVO = "Qwen/Qwen3-1.7B"
MODELO_LLAMA = "unsloth/Llama-3.2-1B-Instruct"
USAR_GEMMA = False

BATCH_SIZE = None
THINKING_EXPLORACION = "both"  # "off", "on" o "both"
THINKING_PILOTO = "off"        # Elígelo después de revisar la exploración.

# Protección: déjalo en False hasta revisar la exploración (sección 7).
EJECUTAR_PILOTO = False

# ------------------- No hace falta editar desde aquí -------------------
import json
import subprocess

opcionales = []
if USAR_GEMMA:
    opcionales.append("google/gemma-3-1b-it")
MODELOS_EXPLORACION = [MODELO_OBJETIVO, MODELO_LLAMA, *opcionales]
MODELOS_PILOTO = list(MODELOS_EXPLORACION)

try:
    from google.colab import userdata

    _token = userdata.get("HF_TOKEN")
except Exception:
    _token = None
if _token:
    os.environ["HF_TOKEN"] = _token
    print("HF_TOKEN disponible.")
else:
    print("Sin HF_TOKEN. Los modelos abiertos funcionan igual.")
    if opcionales:
        print("AVISO: Gemma fallará al cargar y se saltará.")
del _token


def correr(*argumentos):
    """Ejecuta un script del proyecto y muestra su salida mientras avanza."""
    comando = [sys.executable, "-u", *map(str, argumentos)]
    proceso = subprocess.Popen(
        comando, cwd=CODIGO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for linea in proceso.stdout:
        print(linea, end="")
    return proceso.wait()


def extra_lote():
    return [] if BATCH_SIZE is None else ["--batch-size", BATCH_SIZE]


print("Modelos exploración:", MODELOS_EXPLORACION)
print("Modelos piloto:", MODELOS_PILOTO)

## 5. Prueba de humo

Corre 9 instancias (una por celda), 1 muestra, con cada modelo, y deja la salida en el disco local. La primera vez descarga los modelos (unos 7 GB en total). Al final imprime un rollout real completo por modelo: revísalos para ver que cada uno llama a la herramienta y cierra con `<answer>`.

In [ ]:
HUMO = Path("/content/humo")
shutil.rmtree(HUMO, ignore_errors=True)
codigo = correr(
    "scripts/run_pilot.py", "--stage", "exploracion", "--models", *MODELOS_EXPLORACION,
    "--limit", 9, "--n-samples", 1, "--out", HUMO, *extra_lote(),
)
print("Código de salida:", codigo)

archivos = sorted(HUMO.glob("rollouts_*.jsonl"))
if not archivos:
    print("No hay rollouts. Lee", HUMO / "manifest.json")
    print((HUMO / "manifest.json").read_text(encoding="utf-8"))
for archivo in archivos:
    filas = [json.loads(l) for l in archivo.read_text(encoding="utf-8").splitlines() if l.strip()]
    fila = next((f for f in filas if f["condition"] == "tools"), filas[0])
    print("#" * 30, fila["model"], "#" * 30)
    print("=" * 30, "PROMPT", "=" * 30)
    print(fila["prompt"])
    for mensaje in fila["messages"]:
        print("=" * 30, mensaje["role"].upper(), "=" * 30)
        print(mensaje["content"])
    print("=" * 30, "VEREDICTO", "=" * 30)
    print(json.dumps(fila["verdict"], ensure_ascii=False, indent=2))
    print("Uso:", fila["usage"])
    print(f"Aciertos en el humo: {sum(f['verdict']['reward'] == 1.0 for f in filas)}/{len(filas)}")


## 6. Exploración (etapa 4 del plan)

5 instancias por celda, 2 muestras, ambas condiciones: 180 rollouts por variante. Con `THINKING_EXPLORACION = "both"`, Qwen corre con y sin thinking; Llama no tiene ese modo y corre sin él. Usa la semilla 11 y el split `train`, distintos del piloto, para que calibrar aquí no contamine el piloto.

Estimación a priori, **no medida**: entre 30 y 90 minutos en una T4, incluidas las descargas. Las líneas `TOTAL ...` muestran la estimación real tras los primeros lotes. Se puede volver a ejecutar esta celda: continúa donde quedó.

Con 2 muestras no existe pass@5, y con 5 instancias por celda los intervalos son muy anchos: sirve para detectar fallas, no para decidir.

In [ ]:
DIR_EXPLORACION = RESULTADOS / "exploracion"
codigo = correr(
    "scripts/run_pilot.py", "--stage", "exploracion", "--models", *MODELOS_EXPLORACION,
    "--thinking", THINKING_EXPLORACION, "--out", DIR_EXPLORACION, *extra_lote(),
)
print("Código de salida:", codigo)
correr("scripts/analyze_results.py", DIR_EXPLORACION)

In [ ]:
from IPython.display import Markdown, display

display(Markdown((DIR_EXPLORACION / "report.md").read_text(encoding="utf-8")))

## 7. Piloto (etapa 5 del plan)

**Corre esto solo después de revisar la exploración.** Son 40 instancias por celda, 5 muestras y 2 condiciones: **3.600 rollouts por variante**, en el split `eval` con semilla 2026.

Puede tomar varias horas por modelo y superar una sesión gratuita de Colab. La celda siguiente calcula el tiempo con lo medido en tu exploración. Si la sesión se corta, repite las secciones 1 a 4 y esta sección: continuará sin duplicar.

Para ejecutar: pon `EJECUTAR_PILOTO = True` en la sección 4, ejecuta esa celda y luego estas.

In [ ]:
def estimar_piloto(rollouts_por_variante=3600):
    manifiesto = DIR_EXPLORACION / "manifest.json"
    if not manifiesto.exists():
        print("Sin exploración: no hay tiempos medidos.")
        return
    estados = json.loads(manifiesto.read_text(encoding="utf-8")).get("modelos", {})
    total = 0.0
    for etiqueta, estado in estados.items():
        por_rollout = estado.get("segundos_por_rollout")
        if por_rollout is None:
            print(f"{etiqueta}: sin medición ({estado.get('estado')})")
            continue
        horas = por_rollout * rollouts_por_variante / 3600
        print(f"{etiqueta}: {por_rollout:.2f} s/rollout -> ~{horas:.1f} h para 3.600 rollouts")
        total += horas
    print(f"Si corrieras todas las variantes anteriores: ~{total:.1f} h")
    print("Elige en la sección 4 solo las que necesites; la exploración incluye descargas.")


estimar_piloto()

In [ ]:
DIR_PILOTO = RESULTADOS / "piloto"
if EJECUTAR_PILOTO:
    codigo = correr(
        "scripts/run_pilot.py", "--stage", "piloto", "--models", *MODELOS_PILOTO,
        "--thinking", THINKING_PILOTO, "--out", DIR_PILOTO, *extra_lote(),
    )
    print("Código de salida:", codigo)
else:
    print("Piloto no ejecutado: EJECUTAR_PILOTO = False en la sección 4.")

## 8. Análisis y entrega

Esta celda analiza las etapas que existan y crea **`MyDrive/inventory_harness/resultados_inventory_harness.zip`**.

**Ese archivo es el que debes devolvernos.** Trae rollouts, `manifest.json`, `report.md`, `summary.json` y `muestra_errores.jsonl` de cada etapa. Sirve también con una corrida incompleta.

In [ ]:
import datetime

ZIP_RESULTADOS = BASE / "resultados_inventory_harness.zip"
local = Path("/content/resultados_inventory_harness.zip")
etapas = [d for d in (RESULTADOS / "exploracion", RESULTADOS / "piloto") if (d / "manifest.json").exists()]
if not etapas:
    print("Todavía no hay resultados.")
else:
    for carpeta in etapas:
        correr("scripts/analyze_results.py", carpeta)
    with zipfile.ZipFile(local, "w", zipfile.ZIP_DEFLATED) as archivo:
        for carpeta in etapas:
            for ruta in sorted(carpeta.iterdir()):
                if ruta.is_file() and not ruta.name.endswith(".tmp"):
                    archivo.write(ruta, f"{carpeta.name}/{ruta.name}")
    shutil.copy(local, ZIP_RESULTADOS)
    print(datetime.datetime.now().isoformat(timespec="seconds"))
    print(f"LISTO. Devuélvenos este archivo ({local.stat().st_size / 2**20:.1f} MiB):")
    print(ZIP_RESULTADOS)

Si prefieres descargarlo directamente a tu computador, ejecuta esta celda. La copia en Drive ya está hecha.

In [ ]:
from google.colab import files

files.download(str(local))